# NSynth Flow Matching Speedrun

Train a pitch-conditioned flow matching model that generates musical instrument sounds from scratch — in **~1 hour on a free Colab T4 GPU**.

**Before you begin**: Runtime → Change runtime type → **T4 GPU**

---

**What you'll do:**
1. Download a subset of the NSynth dataset (~1.4 GB)
2. Train a UNet flow matching model (~213k parameters)
3. Evaluate with Fréchet Distance and pitch class accuracy
4. Generate and listen to audio samples

**Expected results on T4 (~50–60 min training):**
- FD ≈ 270–310 (random baseline: ~15 000)
- Pitch class accuracy ≈ 80–95% (random baseline: 8.3%)

## 1 · Install dependencies

In [ ]:
# torch + torchaudio are pre-installed on Colab with CUDA support
# We just need librosa and sklearn (scikit-learn is already present, but librosa may not be)
!pip install -q librosa

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

## 2 · Clone the repo

In [ ]:
# ⚠️  Replace with your own repo URL after pushing to GitHub
REPO_URL = "https://github.com/YOUR_USERNAME/nsynth-speedrun"

import os
if not os.path.exists("/content/nsynth-speedrun"):
    !git clone {REPO_URL} /content/nsynth-speedrun

%cd /content/nsynth-speedrun
!ls

## 3 · Download NSynth data

We download two splits:
- **nsynth-valid** (~1.4 GB, 12k files) — used for **training**
- **nsynth-test** (~330 MB, 4k files) — used for **evaluation**

> **Why not nsynth-train?** The training split is 22 GB, which is impractical for Colab.
> Using nsynth-valid + keyboard instrument filter gives ~1 000 keyboard files.
> We compensate with more epochs (1 500) to reach ~13 500 gradient steps,
> matching the convergence of larger-data runs.
>
> If you have access to nsynth-train (e.g. via Google Drive), pass
> `--train_dir /path/to/nsynth-train/audio --max_files 5000 --epochs 500` instead.

In [ ]:
import os
DATA_ROOT = "/content/nsynth"
os.makedirs(DATA_ROOT, exist_ok=True)

TRAIN_DIR = f"{DATA_ROOT}/nsynth-valid/audio"
TEST_DIR  = f"{DATA_ROOT}/nsynth-test/audio"

# Download nsynth-test (~330 MB, ~2 min)
if not os.path.exists(TEST_DIR):
    print("Downloading nsynth-test (~330 MB)...")
    !wget -q http://download.magenta.tensorflow.org/datasets/nsynth/nsynth-test.jsonwav.tar.gz \
           -O /tmp/nsynth-test.tar.gz
    !tar -xf /tmp/nsynth-test.tar.gz -C {DATA_ROOT}
    !rm /tmp/nsynth-test.tar.gz
    print("Test set ready.")
else:
    print("nsynth-test already present.")

# Download nsynth-valid (~1.4 GB, ~5 min)
if not os.path.exists(TRAIN_DIR):
    print("Downloading nsynth-valid (~1.4 GB)...")
    !wget -q http://download.magenta.tensorflow.org/datasets/nsynth/nsynth-valid.jsonwav.tar.gz \
           -O /tmp/nsynth-valid.tar.gz
    !tar -xf /tmp/nsynth-valid.tar.gz -C {DATA_ROOT}
    !rm /tmp/nsynth-valid.tar.gz
    print("Valid set ready.")
else:
    print("nsynth-valid already present.")

In [ ]:
# Verify data and count keyboard files available for training
import glob
all_train = glob.glob(f"{TRAIN_DIR}/*.wav")
keyboard_train = [f for f in all_train if os.path.basename(f).startswith("keyboard")]
all_test  = glob.glob(f"{TEST_DIR}/*.wav")

print(f"Train (valid split): {len(all_train):,} total WAVs")
print(f"  of which keyboard: {len(keyboard_train):,}")
print(f"Test:                {len(all_test):,} WAVs")

## 4 · Train

**Winning recipe** (Exp V): UNet + Muon optimizer + logit-normal timestep sampling + bf16 autocast.

We use `--instrument_filter keyboard` to restrict training to keyboard sounds — this simplifies the task and dramatically improves quality. With ~1 000 keyboard files in the valid split, we run more epochs to compensate for fewer files.

| Setting | Value | Why |
|---|---|---|
| `--model_type unet` | UNet 2D (213k params) | Best FD/pitch accuracy |
| `--optimizer muon --lr 0.02` | Muon | ~2× faster convergence than AdamW |
| `--t_sample logit_normal` | SD3/Flux trick | Focuses training on hard mid-noise timesteps |
| `--instrument_filter keyboard` | Keyboard sounds only | Easier task → much better FD |
| `--bf16` | bfloat16 autocast | Free −15% training time on modern GPUs |

> **T4 note**: T4 does not natively support bfloat16 — the `--bf16` flag will print a warning and fall back to fp32 automatically.

In [ ]:
# Count keyboard files to set epochs automatically
import glob, os
keyboard_files = glob.glob(f"{TRAIN_DIR}/keyboard_*.wav")
n_keyboard = min(len(keyboard_files), 5000)
batch_size = 128
target_steps = 15_000  # ~19k for full recipe; 15k is a good Colab budget
steps_per_epoch = max(1, n_keyboard // batch_size)
epochs = max(100, target_steps // steps_per_epoch)
print(f"Keyboard files: {n_keyboard}")
print(f"Steps/epoch: {steps_per_epoch}")
print(f"Epochs to reach ~{target_steps} steps: {epochs}")

In [ ]:
CHECKPOINT = "/content/flow_model.pt"

!CUDA_VISIBLE_DEVICES=0 python train.py \
  --model_type unet \
  --instrument_filter keyboard \
  --t_sample logit_normal \
  --max_files 5000 \
  --optimizer muon --lr 0.02 \
  --batch_size 128 \
  --epochs {epochs} \
  --bf16 \
  --train_dir {TRAIN_DIR} \
  --save_path {CHECKPOINT}

## 5 · Evaluate

Two metrics:
- **Fréchet Distance (FD)**: distance between real and generated spectrogram distributions (lower is better; random ≈ 15 000)
- **Pitch class accuracy**: does the model generate C when asked for C? (random baseline = 8.3%)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python eval.py \
  --checkpoint {CHECKPOINT} \
  --test_dir {TEST_DIR} \
  --guidance_scales 1 6

## 6 · Generate audio samples

Generate keyboard sounds across one octave (C4–B4) with Classifier-Free Guidance at scale 6.

In [ ]:
SAMPLE_DIR = "/content/samples"

!CUDA_VISIBLE_DEVICES=0 python infer.py \
  --checkpoint {CHECKPOINT} \
  --pitches 60 61 62 63 64 65 66 67 68 69 70 71 \
  --n_per_pitch 2 \
  --guidance_scale 6.0 \
  --out_dir {SAMPLE_DIR}

In [ ]:
# Listen to generated samples
import IPython.display as ipd
from pathlib import Path

wavs = sorted(Path(SAMPLE_DIR).glob("*.wav"))
print(f"Generated {len(wavs)} audio files.  Playing the first 6:\n")

for wav in wavs[:6]:
    print(wav.name)
    display(ipd.Audio(str(wav)))

## 7 · (Optional) Random baseline

Compare against an untrained model to confirm the model actually learned something.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python eval.py \
  --checkpoint {CHECKPOINT} \
  --test_dir {TEST_DIR} \
  --guidance_scales 1 \
  --random_baseline

## 8 · Things to try

Edit the training cell above and change one variable at a time:

| Experiment | Change | Expected effect |
|---|---|---|
| More epochs | `epochs = 2000` | Lower FD, better pitch accuracy |
| Standard optimizer | `--optimizer adamw --lr 3e-4` | Slower convergence — compare FD at same steps |
| Uniform timesteps | `--t_sample uniform` | Slower convergence vs. logit_normal |
| Smaller model | `--unet_hidden 24` | 125k params, ~81% pitch acc — good on CPU |
| All instruments | remove `--instrument_filter keyboard` | Harder task: model must learn 11 families |
| Higher CFG | `--guidance_scales 1 3 6 10` | See quality/diversity tradeoff |
| More ODE steps | `python infer.py ... --n_steps 200` | Cleaner audio, slower generation |
| Heun's method | Modify `infer.py` to use 2nd-order steps | Same NFE budget, better quality |

---

**Metrics to report:**
- FD@1 (no guidance) vs FD@6 (with guidance) — quantifies how much CFG helps
- Pitch class accuracy — is the model pitch-conditional?
- Training loss curve — does it converge?
- Training time — what's the compute cost of each change?